In [29]:
import os
import sys
import pandas as pd
import numpy as np
import warnings
from collections import Counter
from sklearn.impute import SimpleImputer
warnings.filterwarnings('ignore')

dir = "/Users/julianguyen/Documents/multimodal-pmlb"
os.chdir(dir)

np.random.seed(101)

In [30]:
# load in files
atc = pd.read_csv("data/procdata/files/atac_gene.csv", index_col=0).T
rna = pd.read_csv("data/procdata/files/rna_gene.csv", index_col=0).T
cnv = pd.read_csv("data/procdata/files/cnv.csv", index_col=0).T
mut = pd.read_csv("data/procdata/files/mut.csv", index_col=0).T

# impute and log transform gene counts
imputer = SimpleImputer(strategy='median')
rna_imputed = pd.DataFrame(imputer.fit_transform(rna), index=rna.index, columns=rna.columns)
rna = np.log1p(rna_imputed.clip(lower=0))

# get targest
targets = pd.read_csv("data/procdata/files/meta.csv").set_index("PMLB_organoidID").loc[:, ["doubling_rate", "primary_tumor_site"]]
dt = targets["doubling_rate"]

# get indices
assert atc.index.equals(rna.index)
assert atc.index.equals(cnv.index)
assert atc.index.equals(mut.index)
assert atc.index.equals(targets.index)

## Prediction Task Agnostic Feature Selection

### Filtering low variance genes

In [31]:
# keep top high 10000 variance genes
def filter_low_var_genes(mat, ngenes=10000):
    gene_var = mat.var(axis=0)
    filtered = mat[gene_var.sort_values(ascending=False).head(ngenes).index]
    return filtered

# number of genes before
print("Number of gene features before filtering and subsetting")
print("No. genes in ATAC:", len(atc.columns))
print("No. genes in RNA:", len(rna.columns))
print("No. genes in CNV:", len(cnv.columns))
print("No. genes in MUT:", len(mut.columns))

# filter out low variance genes
af = filter_low_var_genes(atc)
rf = filter_low_var_genes(rna)
cf = filter_low_var_genes(cnv)
mf = filter_low_var_genes(mut)

Number of gene features before filtering and subsetting
No. genes in ATAC: 26658
No. genes in RNA: 25247
No. genes in CNV: 25128
No. genes in MUT: 12316


### Keep genes in >1 omics profile

In [32]:
# keep gene features in >1 omics dataframe
all_genes = list(af.columns) + list(rf.columns) + list(cf.columns) + list(mf.columns)
gene_counts = Counter(all_genes)
keep_genes = [gene for gene, count in gene_counts.items() if count >= 2]
print("\nGenes in >1 omics:", len(keep_genes))

# keep gene features in ALL (4) omics dataframes
common_genes = (
    set(af.columns)
    & set(rf.columns)
    & set(cf.columns)
    & set(mf.columns)
)
print("Common genes across all omics:", len(common_genes))

# subset dataframes
atc = atc[[col for col in atc.columns if col in keep_genes]]
rna = rna[[col for col in rna.columns if col in keep_genes]]
cnv = cnv[[col for col in cnv.columns if col in keep_genes]]
mut = mut[[col for col in mut.columns if col in keep_genes]]

print("\nNumber of genes after subset1")
print("No. genes in ATAC:", len(atc.columns))
print("No. genes in RNA:", len(rna.columns))
print("No. genes in CNV:", len(cnv.columns))
print("No. genes in MUT:", len(mut.columns))

atc.to_csv("data/procdata/model_cg1/atc.csv", index=True)
rna.to_csv("data/procdata/model_cg1/rna.csv", index=True)
cnv.to_csv("data/procdata/model_cg1/cnv.csv", index=True)
mut.to_csv("data/procdata/model_cg1/mut.csv", index=True)
targets.to_csv("data/procdata/model_cg1/meta.csv", index=True)

# subset dataframes
atc = atc[[col for col in atc.columns if col in common_genes]]
rna = rna[[col for col in rna.columns if col in common_genes]]
cnv = cnv[[col for col in cnv.columns if col in common_genes]]
mut = mut[[col for col in mut.columns if col in common_genes]]

print("\nNumber of genes after subset2")
print("No. genes in ATAC:", len(atc.columns))
print("No. genes in RNA:", len(rna.columns))
print("No. genes in CNV:", len(cnv.columns))
print("No. genes in MUT:", len(mut.columns))

atc.to_csv("data/procdata/model_cg2/atc.csv", index=True)
rna.to_csv("data/procdata/model_cg2/rna.csv", index=True)
cnv.to_csv("data/procdata/model_cg2/cnv.csv", index=True)
mut.to_csv("data/procdata/model_cg2/mut.csv", index=True)
targets.to_csv("data/procdata/model_cg2/meta.csv", index=True)


Genes in >1 omics: 9501
Common genes across all omics: 524

Number of genes after subset1
No. genes in ATAC: 8338
No. genes in RNA: 8367
No. genes in CNV: 8890
No. genes in MUT: 6791

Number of genes after subset2
No. genes in ATAC: 524
No. genes in RNA: 524
No. genes in CNV: 524
No. genes in MUT: 524
